# 📖 Tutorial 05: Experiments & Customization

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lessen-xu/Hybrid-Chess/blob/main/notebooks/05_experiments.ipynb)

---

## Who is this for?

You've mastered the AlphaZero pipeline (Tutorials 01-04). Now it's time to **experiment**. This tutorial shows you how to use Hybrid Chess as a research testbed — customizing rules, reward signals, network architectures, and training curricula.

**What you'll learn:**
1. **VariantConfig ablation** — How do game rules affect training?
2. **Reward shaping** — Adding intermediate rewards to speed up learning
3. **Custom network architectures** — Designing your own neural network
4. **Curriculum learning** — Training from simple to complex positions
5. **Experiment tracking** — Logging and comparing runs
6. **Open research questions** — What hasn't been explored yet

**Prerequisites:** Tutorials 01-04.

---

## 0. Setup

In [ ]:
# Uncomment for Google Colab:
# !pip install -q git+https://github.com/lessen-xu/Hybrid-Chess.git

In [ ]:
import random, time
import numpy as np
from collections import Counter

from hybrid.core.env import HybridChessEnv, GameState
from hybrid.core.config import VariantConfig, BOARD_H, BOARD_W
from hybrid.core.types import Side, PieceKind
from hybrid.core.render import render_board
from hybrid.agents.random_agent import RandomAgent
from hybrid.agents.greedy_agent import GreedyAgent
from hybrid.agents.alphabeta_agent import AlphaBetaAgent, SearchConfig
print("✅ Ready")

---

## 1. VariantConfig Ablation

### 1.1 Theory: Why rule variants matter for research

One of the most powerful features of Hybrid Chess as a research testbed is `VariantConfig` — a frozen dataclass that lets you toggle game rules without changing any code.

This enables **controlled experiments**:
- How does the Queen affect Chess's strength? → Remove it with `no_queen=True`
- Are Cannons overpowered? → Add an extra with `extra_cannon=True`
- What if Pawns promote differently? → Disable with `no_promotion=True`

By training separate AlphaZero models under different variants, you can measure the **impact of each rule** on game balance, learning speed, and playing style.

### 1.2 Available flags

| Flag | Default | Effect |
|------|---------|--------|
| `no_queen` | False | Remove Chess Queen from starting position |
| `no_bishop` | False | Remove a Chess Bishop |
| `one_rook` | False | Remove one of Chess's Rooks |
| `extra_cannon` | False | Add a third Cannon for Xiangqi |
| `no_promotion` | False | Pawns can't promote at the far rank |

### 1.3 Designing a balance experiment

To test whether a rule change helps game balance:
1. Define your variants
2. Play many games between equal-strength agents under each variant
3. Compare win rates — the closer to 50/50, the more balanced the game

In [ ]:
# Balance experiment: different variants

def play_balance_test(variant_config, n_games=20, max_ply=150):
    """Play games between two Greedy agents to measure balance."""
    results = Counter()
    for i in range(n_games):
        env = HybridChessEnv(config=variant_config, max_plies=max_ply)
        state = env.reset()
        chess_agent = GreedyAgent()
        xiangqi_agent = GreedyAgent()
        agents = {Side.CHESS: chess_agent, Side.XIANGQI: xiangqi_agent}
        info = None
        while True:
            legal = env.legal_moves()
            if not legal:
                break
            move = agents[state.side_to_move].select_move(state, legal)
            state, _, done, info = env.step(move)
            if done:
                break
        if info and info.winner == Side.CHESS:
            results['chess'] += 1
        elif info and info.winner == Side.XIANGQI:
            results['xiangqi'] += 1
        else:
            results['draw'] += 1
    return results

# Define variants to test
variants = {
    'Standard':     VariantConfig(),
    'No Queen':     VariantConfig(no_queen=True),
    'Extra Cannon': VariantConfig(extra_cannon=True),
    'No Promote':   VariantConfig(no_promotion=True),
}

N = 20
print(f"Balance experiment ({N} games per variant, Greedy vs Greedy):")
print(f"{'Variant':<16} {'Chess':>6} {'Xiangqi':>8} {'Draw':>5} {'Balance':>8}")
print("─" * 48)

for name, cfg in variants.items():
    r = play_balance_test(cfg, n_games=N)
    total = r['chess'] + r['xiangqi'] + r['draw']
    decisive = r['chess'] + r['xiangqi']
    balance = r['chess'] / decisive * 100 if decisive > 0 else 50.0
    print(f"{name:<16} {r['chess']:>6} {r['xiangqi']:>8} {r['draw']:>5} {balance:>6.0f}% C")

print(f"\n💡 50% = perfectly balanced. Deviations show which side benefits.")
print(f"   Removing the Queen weakens Chess → Xiangqi should win more.")

---

## 2. Reward Shaping

### 2.1 Theory: The sparse reward problem

In standard AlphaZero, the **only feedback** is the game outcome: +1 (win), −1 (loss), or 0 (draw). This means the agent doesn't get any signal until the game is **completely over** — which could be 200 moves later.

This is called the **sparse reward problem** and it makes learning slow, especially for:
- Long games
- Games where early actions have delayed effects
- Novel game designs where the network has no prior knowledge

### 2.2 Reward shaping: Adding intermediate signals

**Reward shaping** adds extra reward signals to guide learning:

```
Original AlphaZero:   z = game_outcome (+1 / -1 / 0)
With reward shaping:  z = game_outcome + Σ r_shaping(state, move)
```

Our framework supports this through the `reward_shaper` hook in `SelfPlayConfig`:

```python
# Custom reward shaper: bonus for captures
def capture_bonus(state, move, next_state):
    captured = state.board.get(move.tx, move.ty)
    if captured:
        return 0.01 * PIECE_VALUES[captured.kind]  # Scale: tiny vs game outcome
    return 0.0

cfg = SelfPlayConfig(reward_shaper=capture_bonus)
```

### 2.3 Design considerations

| Principle | Explanation |
|-----------|------------|
| **Scale** | Shaping rewards should be ≪ game outcome (±1). Typically 0.001–0.05 |
| **Potential-based** | Ideally, shaping = Φ(s') − Φ(s) for some potential function Φ. This preserves optimal policy (reward shaping theorem) |
| **Diminishing** | Can anneal the shaping reward to 0 over training, letting pure outcome dominate |
| **Risk** | Bad shaping can teach wrong strategies (e.g., hoarding material instead of checkmating) |

In [ ]:
from hybrid.rl.az_selfplay import SelfPlayConfig, PIECE_VALUES

# Example reward shapers

def capture_bonus_shaper(state, move, next_state):
    """Small bonus for capturing pieces."""
    captured = state.board.get(move.tx, move.ty)
    if captured:
        return 0.01 * PIECE_VALUES.get(captured.kind, 1.0)
    return 0.0

def center_control_shaper(state, move, next_state):
    """Bonus for moving pieces toward the center."""
    # Center of a 9x10 board ≈ (4, 4.5)
    old_dist = abs(move.fx - 4) + abs(move.fy - 4.5)
    new_dist = abs(move.tx - 4) + abs(move.ty - 4.5)
    return 0.002 * (old_dist - new_dist)  # positive if moving closer

def aggression_shaper(state, move, next_state):
    """Combined: capture bonus + center control + check bonus."""
    r = capture_bonus_shaper(state, move, next_state)
    r += center_control_shaper(state, move, next_state)
    return r

print("Three example reward shapers:")
print("  1. capture_bonus — +0.01×piece_value for captures")
print("  2. center_control — +0.002 for moving toward center")
print("  3. aggression — combination of both")
print()

# Show how to use them
print("Usage:")
print("  cfg = SelfPlayConfig(reward_shaper=capture_bonus_shaper)")
print("  # Then pass cfg to self_play_game()")
print()

# Demonstrate on a sample move
env = HybridChessEnv()
state = env.reset()
legal = env.legal_moves()
mv = legal[0]  # first legal move
piece = state.board.get(mv.fx, mv.fy)
r_cap = capture_bonus_shaper(state, mv, None)
r_ctr = center_control_shaper(state, mv, None)
print(f"Sample: {piece.kind.name} ({mv.fx},{mv.fy})→({mv.tx},{mv.ty})")
print(f"  capture_bonus:  {r_cap:.4f}")
print(f"  center_control: {r_ctr:.4f}")

---

## 3. Custom Network Architectures

### 3.1 Theory: Architecture as a hyperparameter

The default `PolicyValueNet` uses 64 channels and 3 residual blocks. But the optimal architecture depends on the problem:

| Parameter | Trade-off |
|-----------|----------|
| **More channels** (128, 256) | Better representation capacity, but slower training+inference |
| **More res blocks** (5, 10, 20) | Deeper reasoning, but harder to train, diminishing returns |
| **Fewer channels** (32) | Faster, but may underfit on complex positions |

For research, you can subclass `BaseModel` to create entirely different architectures:

```python
from hybrid.rl.az_network import BaseModel

class MyCustomNet(BaseModel):
    def __init__(self):
        super().__init__()
        # ... your custom layers ...
    
    def forward(self, x):
        # Must return: (policy_logits, value)
        # policy_logits shape: (B, 92, 10, 9)
        # value shape: (B, 1)
        return policy_logits, value
```

### 3.2 Experiment ideas

- **Tiny net** (32 channels, 1 res block): How good can a small model get? Useful for understanding what the network learns.
- **Wide net** (256 channels, 3 res blocks): More capacity — does it help or just waste compute?
- **Deep net** (64 channels, 10 res blocks): Can deeper networks "think" further ahead?
- **Attention layers**: Replace conv blocks with self-attention to capture long-range piece interactions.

In [ ]:
import torch
from hybrid.rl.az_network import PolicyValueNet
from hybrid.rl.az_encoding import NUM_STATE_CHANNELS

# Compare network sizes
configs = [
    ("Tiny",     32, 1),
    ("Default",  64, 3),
    ("Wide",    128, 3),
    ("Deep",     64, 6),
    ("Large",   128, 6),
]

print(f"{'Name':<10} {'Channels':>9} {'Blocks':>7} {'Params':>12} {'FWD (ms)':>10}")
print("─" * 52)

x = torch.randn(1, NUM_STATE_CHANNELS, BOARD_H, BOARD_W)

for name, ch, blocks in configs:
    net = PolicyValueNet(in_channels=NUM_STATE_CHANNELS, num_channels=ch, num_res_blocks=blocks)
    params = sum(p.numel() for p in net.parameters())
    
    # Time forward pass
    net.eval()
    with torch.no_grad():
        t0 = time.time()
        for _ in range(10):
            net(x)
        dt = (time.time() - t0) / 10 * 1000
    
    print(f"{name:<10} {ch:>9} {blocks:>7} {params:>12,} {dt:>9.1f}ms")

print(f"\n💡 The default (64ch/3blocks/~300K params) is a good starting point.")
print(f"   For faster iteration, try Tiny. For maximum strength, try Large.")

---

## 4. Curriculum Learning

### 4.1 Theory: Learning simple things first

Humans learn chess by first studying **simple endgames** (K+R vs K) before tackling full games. AlphaZero can benefit from the same approach.

**Curriculum learning** gradually increases task difficulty:

```
Stage 1: Endgames (3-5 pieces, ~10 moves to mate)
  ↓
Stage 2: Middlegames (10-15 pieces, complex tactics)
  ↓
Stage 3: Full games (all pieces, from opening)
```

### 4.2 Benefits

- **Faster value learning**: Endgames have clear right/wrong evaluations
- **Checkmate patterns**: The network learns to deliver checkmate before learning openings
- **Reward density**: Short games provide more outcome signal per step

### 4.3 Implementation: Endgame spawner

Our framework includes an endgame position spawner for curriculum training. You can also set up custom positions using FEN notation:

In [ ]:
from hybrid.core.fen import parse_fen, board_to_fen

# Custom endgame position via FEN
# Example: Chess has King + Rook, Xiangqi has General only
env = HybridChessEnv()
state = env.reset()

# Show the FEN of opening position
fen = board_to_fen(state.board, state.side_to_move)
print(f"Opening FEN: {fen}")
print()

# Load a custom position
fen_endgame = "9/4g4/9/9/9/9/9/9/9/4K3R c"  # K+R vs General
try:
    board, side = parse_fen(fen_endgame)
    state_eg = env.reset_from_fen(fen_endgame)
    print(f"Custom endgame loaded:")
    print(render_board(state_eg.board))
except Exception as e:
    print(f"FEN loading (demo): {e}")
    print(f"⬆️ This may fail if the FEN doesn't match the board validator.")
    print(f"   FEN format: piece_positions side_to_move")

print(f"\n💡 Use FEN positions for:")
print(f"   • Curriculum learning (start from endgames)")
print(f"   • Debugging (reproduce specific positions)")
print(f"   • Puzzle solving (set up tactical problems)")

---

## 5. Experiment Tracking

### 5.1 What to log

For any serious experiment, track these metrics:

| Category | Metrics |
|----------|--------|
| **Loss** | policy_loss, value_loss, total_loss per training step |
| **Self-play** | avg game length, win rates (chess/xiangqi/draw), resign rate |
| **Gating** | acceptance rate, head-to-head win rate |
| **Evaluation** | win rate vs Random, Greedy, AB-d1, AB-d2, previous checkpoint |
| **Timing** | self-play time, training time, evaluation time per iteration |

### 5.2 Tools

```python
# TensorBoard (built-in)
from torch.utils.tensorboard import SummaryWriter
writer = SummaryWriter('runs/experiment_1')
writer.add_scalar('loss/policy', policy_loss, step)
writer.add_scalar('loss/value', value_loss, step)

# WandB (optional, requires account)
import wandb
wandb.init(project='hybrid-chess', name='exp_1')
wandb.log({'policy_loss': policy_loss, 'value_loss': value_loss})
```

### 5.3 Comparing experiments

When comparing two training runs, the most important metric is **win rate against a fixed opponent** (e.g., AlphaBeta d=2). This directly measures playing strength, unlike loss values which can be misleading.

---

## 6. Open Research Questions

### 6.1 Things nobody has tried yet

Hybrid Chess is a **novel game** — much of its strategy space is unexplored. Here are some research directions:

| Question | Approach |
|----------|----------|
| **Is the game balanced?** | Train strong agents for both sides, measure win rate |
| **What's the optimal opening?** | Analyze MCTS visit distributions in the first 10 moves |
| **Does the Cannon dominate?** | Compare `extra_cannon` vs standard → if XQ wins much more, Cannon is key |
| **Can smaller nets match bigger nets?** | Train 32ch vs 128ch for same compute budget |
| **Does reward shaping help or hurt long-term?** | Train with/without shaping for 200 iterations |
| **Transfer learning?** | Pre-train on Chess/Xiangqi, fine-tune on Hybrid Chess |
| **Asymmetric architectures?** | Different networks for each side (they have different pieces/rules) |

### 6.2 Getting started on a research project

1. **Pick one variable** to study (architecture, reward, variant, curriculum)
2. **Define your control** (default training settings)
3. **Change one thing** at a time
4. **Run enough iterations** (at least 50-100) to see meaningful trends
5. **Log everything** — you'll want to analyze results later
6. **Compare against baselines** — not just loss curves, but actual playing strength

---

## 🎓 Summary

| Topic | Key takeaway |
|-------|-------------|
| **VariantConfig** | Frozen dataclass for rule ablation without code changes |
| **Balance testing** | Play N games under variant, compare win rates |
| **Reward shaping** | Add intermediate signals via `reward_shaper` hook; keep scale small |
| **Custom architectures** | Subclass `BaseModel`; return `(policy_logits, value)` |
| **Curriculum learning** | Start from endgames (via FEN), graduate to full games |
| **Experiment tracking** | TensorBoard/WandB; focus on win rate vs fixed opponent |
| **Research directions** | Game balance, optimal openings, architecture search, transfer learning |

## 🏁 You've completed all tutorials!

You now understand:
1. ✅ The game environment and its API (Tutorial 01)
2. ✅ Classical search: Minimax, Alpha-Beta, evaluation functions (Tutorial 02)
3. ✅ Monte Carlo Tree Search and how AlphaZero enhances it (Tutorial 03)
4. ✅ The full AlphaZero training pipeline (Tutorial 04)
5. ✅ How to run experiments and push the boundaries (Tutorial 05)

**Next steps:**
- 🚀 Run `python -m hybrid train` to start your own training
- 🎮 Play against your model via `python -m hybrid server`
- 📊 Share your results and findings!
- ⭐ Star the repo if you found this useful!